# 크라베오 AI 재학습 — v5 (정체성 데이터 오염 수정)
# Cravveo AI Retrain — v5 (fixed contaminated identity data)

**변경 사항 | Changes:**
- Day 041에서 학습 데이터 오염 발견: 기존 92개 중 약 40개가 크라베오랑 무관한 AI 마케팅 잡담이었고,
  나머지도 "쇼핑몰"(실제로 없음), "Gemma-2B"(Day 026에 Llama 3.2로 교체됨) 같은 오래된/틀린 정보 포함
- 정확한 사실만 담은 20개로 전체 교체 후 HuggingFace에 재업로드 완료
- 이번 학습: 새 정체성 데이터 20개 + Obsidian 데이터 203개 = 총 223개
- Discovered Day 041: ~40 of 92 existing samples were unrelated AI-marketing rambling,
  and the rest had stale/wrong facts ("online shop" that doesn't exist, "Gemma-2B" replaced at Day 026)
- Replaced with 20 verified-accurate identity samples, already re-uploaded to HuggingFace
- This training run: 20 identity samples + 203 Obsidian samples = 223 total

**순서 | Steps:**
1. 설치
2. 모델 로드
3. LoRA 어댑터
4. 데이터셋 로드 + 합치기
5. 학습
6. 테스트
7. HuggingFace 업로드
8. GGUF 변환 (q4_K_M — Day 039에서 검증된 방식)


In [ ]:
# 셀 1: 필수 라이브러리 설치
# Cell 1: Install required libraries
!pip install unsloth trl datasets


In [ ]:
# 셀 2: CUDA 오류 방지 + 모델 로드
# Cell 2: CUDA fix + model load
import ctypes, glob

paths = (
    glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib/libnvJitLink*.so*") +
    glob.glob("/usr/lib/**/libnvJitLink*.so*", recursive=True) +
    glob.glob("/usr/local/cuda*/**/libnvJitLink*.so*", recursive=True)
)
if paths:
    ctypes.CDLL(paths[0], mode=ctypes.RTLD_GLOBAL)
    print(f"CUDA fix 적용 | CUDA fix applied: {paths[0]}")

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=512,
    load_in_4bit=True,
)
print("모델 로드 완료 | Model loaded")


In [ ]:
# 셀 3: LoRA 어댑터 설정
# Cell 3: LoRA adapter setup
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("LoRA 어댑터 적용 완료 | LoRA adapter applied")


In [ ]:
# 셀 4: 데이터셋 로드 + 합치기
# Cell 4: Load datasets + combine
from datasets import load_dataset, Dataset, concatenate_datasets
import json

# --- 정체성 데이터셋 (HuggingFace, Day 041에 20개로 정리됨) ---
existing_raw = load_dataset("cravveo/cravveo-briefing-dataset", split="train")

def convert_existing(example):
    return {"text": f"질문: {example['instruction']}\n답변: {example['output']}"}

existing = existing_raw.map(convert_existing, remove_columns=existing_raw.column_names)
print(f"정체성 데이터셋: {len(existing)}개 (기대값: 20개)")

# --- Obsidian 데이터셋 (파일 업로드) ---
from google.colab import files
print("obsidian_dataset.jsonl 파일을 업로드하세요 | Upload obsidian_dataset.jsonl")
uploaded = files.upload()

new_data = []
with open("obsidian_dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        new_data.append(json.loads(line.strip()))

new_dataset = Dataset.from_list(new_data)
print(f"Obsidian 데이터셋: {len(new_dataset)}개")

# --- 합치기 ---
combined = concatenate_datasets([existing, new_dataset])
print(f"\n합계 | Total: {len(combined)}개")
print("샘플 확인 | Sample check:")
print(combined[0]['text'])
print("---")
print(combined[-1]['text'])


In [ ]:
# 셀 5: 학습
# Cell 5: Training
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=combined,
    dataset_text_field="text",
    max_seq_length=512,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="cravveo_v5_output",
        save_strategy="no",
    ),
)

print("학습 시작 | Training start...")
trainer.train()
print("학습 완료 | Training complete!")


In [ ]:
# 셀 6: 테스트 — 학습 결과 확인
# Cell 6: Quick test
FastLanguageModel.for_inference(model)

test_questions = [
    "크라베오 컴퍼니가 뭐야?",
    "ablescan.app은 뭐야?",
    "지금 수익이 나고 있어?",
]
for q in test_questions:
    inputs = tokenizer([f"질문: {q}\n답변: "], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.2)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("---")


In [ ]:
# 셀 7: HuggingFace 업로드
# Cell 7: Push to HuggingFace
from google.colab import userdata
from huggingface_hub import login

token = userdata.get('HF_TOKEN')
login(token=token)

model.save_pretrained("cravveo-llama-lora-v5")
model.push_to_hub("cravveo/cravveo-llama-lora", token=token)  # 같은 레포에 덮어쓰기
tokenizer.push_to_hub("cravveo/cravveo-llama-lora", token=token)

print("HuggingFace 업로드 완료 | HuggingFace upload complete")
print("https://huggingface.co/cravveo/cravveo-llama-lora")


In [ ]:
# 셀 8: GGUF 변환 (q4_K_M — Day 039에서 검증된 방식)
# Cell 8: GGUF conversion (q4_K_M — proven method from Day 039)
model.save_pretrained_gguf(
    "cravveo-v5-gguf",
    tokenizer,
    quantization_method="q4_k_m"
)

from google.colab import files
import os

folder = "cravveo-v5-gguf_gguf" if os.path.isdir("cravveo-v5-gguf_gguf") else "cravveo-v5-gguf"
gguf_file = [f for f in os.listdir(folder) if f.endswith(".gguf")][0]
size_gb = os.path.getsize(f"{folder}/{gguf_file}") / 1e9

print(f"파일명: {gguf_file}")
print(f"파일 크기: {size_gb:.2f} GB")

files.download(f"{folder}/{gguf_file}")
print("다운로드 시작 | Download started")
